# BRCA1 comparative analysis, Step 0b: what effect can 4 vs 8 donors detect?

A **blind** design check, run before the main test is computed for any BRCA1 donor. It reads the
**8 WT donors only**: no BRCA1 expression, cell classes, identity calls or Δ are loaded, so nothing
here can bias the main test. Companion to
[`brca1-preneoplastic-comparative.ipynb`](brca1-preneoplastic-comparative.ipynb).

**Why.** Step 5 compares per-donor Δ = LP fraction under Arm D − LP fraction under Arm B between
4 BRCA1 and 8 WT donors with an exact permutation test. Two quantities that WT donors alone
determine decide whether that test can find anything:

1. **The SD of Δ between donors** is the noise the test has to beat. The permutation test on a
   difference in means depends only on differences between donors, so its power depends on the
   effect in units of this SD. The SD therefore fixes the minimum detectable effect (MDE).
2. **The rescued share of epithelium, w.** Per donor, Δ = w · (r − p), where p is the Arm B LP
   fraction and r the LP fraction among rescued epithelial cells. Whatever the rescued cells are,
   |Δ| ≤ w, so w caps how large any genotype effect can be and makes the MDE judgeable.

If the MDE exceeds any plausible effect, the Step 5 report says so first and is framed as
**estimation**: the BRCA1 − WT difference in mean Δ with a 95% CI, and for a null result "we can
exclude effects larger than X", instead of a verdict from the p-value. That is the honest framing
for this cohort in any case: the authors' own abundance test (Appendix S1B, P = 0.14) came from the
same 4 BRCA1 donors.

**Output.** `step0b_design_sensitivity/design_decision.json`. Step 5 loads it with
`DesignDecision.from_json` and leads its report with `DesignDecision.summary()`.

**What is provisional.** Steps 1–3 are not built yet, so Δ here uses stand-ins:

| Main pipeline | This notebook | Consequence |
|---|---|---|
| Step 1: PBS classes from frozen WT quantiles | same rule: cutoffs from pooled WT noise cells | identical for WT donors |
| Step 2: identity by mapping onto Arm B clusters | per-cell Lim et al. (2009) signature scores, standardised on Arm B cells only | rescued cells still cannot define LP, but individual calls differ |
| Step 3: ambient filter on rescued LP calls | none | r may be inflated by ambient LP transcripts, so Δ and its SD can be off in either direction |

`ANKRD30A` and `KRT14` (shared ambient core) are dropped from the signatures, since they are
unreliable for mature luminal and basal identity. Once Steps 2–3 exist, re-run this notebook with
their WT labels. It stays blind as long as that happens before any BRCA1 Δ is computed; if the
verdict changes, record both and use the Step 2–3 version.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from signals_in_the_noise.analysis.brca1_cohort import (
    DONORS,
    GENOTYPE_PALETTE,
    Genotype,
    build_cell_table,
    load_or_build_cell_table,
)
from signals_in_the_noise.analysis.cell_identity import (
    AMBIENT_CORE_MARKERS,
    LIM_SIGNATURE_FILES,
    call_cell_types,
    load_lim_signatures,
    score_signatures,
)
from signals_in_the_noise.analysis.design_sensitivity import (
    DesignDecision,
    ceiling_effect,
    fold_enrichment_effect,
    minimum_detectable_effect,
    normal_errors,
    resampled_errors,
    sd_with_ci,
    simulate_ci_coverage,
    simulate_power,
    t_interval_half_width,
)
from signals_in_the_noise.analysis.rescue_arms import (
    ARMS,
    assign_cell_classes,
    delta_by_donor,
    rescued_classes,
    rescued_share_by_donor,
)
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging()

In [ ]:
# Parameters
FORCE_RESCORE = False  # True recomputes signature scores instead of reading the cache
REFERENCE_ARM, TEST_ARM = "B", "D"  # Δ = LP fraction under TEST_ARM - under REFERENCE_ARM
ALPHA = 0.05
ALTERNATIVE = "two-sided"  # as planned for Step 5; "greater" would lower the MDE
TARGET_POWER = 0.80
PLAUSIBLE_FOLD = 2.0  # generous benchmark: BRCA1 rescued epithelial cells LP twice as often as WT's
N_SIMULATIONS = 4000
SEED = 20260927

WT_DONORS = tuple(d for d in DONORS if d.genotype == Genotype.WT)
N_BRCA1_DONORS = sum(d.genotype == Genotype.BRCA1 for d in DONORS)  # group size only; no BRCA1 data is read
SIGNATURE_NAMES = list(LIM_SIGNATURE_FILES)

ANALYSIS_DIR = get_data_path("GSE161529_analysis_cache") / "brca1_preneoplastic_comparative"
DESIGN_DIR = ANALYSIS_DIR / "step0b_design_sensitivity"
DESIGN_DIR.mkdir(parents=True, exist_ok=True)
CELL_TABLE_PATH = ANALYSIS_DIR / "cell_table.csv.gz"  # shared with Step 0
SCORES_PATH = DESIGN_DIR / "wt_signature_scores.csv.gz"


def save_figure(fig: plt.Figure, name: str) -> None:
    fig.savefig(DESIGN_DIR / f"{name}.png", dpi=200, bbox_inches="tight")


print("DESIGN_DIR:", DESIGN_DIR)
print(f"Main test: {N_BRCA1_DONORS} BRCA1 vs {len(WT_DONORS)} WT donors")

## 1. WT cells only

Reuses Step 0's per-cell QC table, which holds only QC metrics that Step 0 already compared between
genotypes. BRCA1 rows are dropped immediately on load, and the assertions stop the notebook if any
survive.

In [ ]:
def _build_from_gse161529() -> pd.DataFrame:
    from signals_in_the_noise.preprocessing.gse161529 import GSE161529

    return build_cell_table(GSE161529().get_dataset)


all_cells = load_or_build_cell_table(CELL_TABLE_PATH, _build_from_gse161529)
cells = all_cells.loc[all_cells["genotype"] == Genotype.WT].reset_index(drop=True)
del all_cells
cells["donor"] = cells["donor"].cat.remove_unused_categories()

assert set(cells["genotype"]) == {Genotype.WT}, "BRCA1 cells must not enter this notebook."
assert list(cells["donor"].cat.categories) == [d.author_id for d in WT_DONORS]
print(f"{len(cells):,} WT barcodes from {cells['donor'].nunique()} donors")

## 2. Cell classes with frozen WT quantiles (Step 1 rule)

PBS cutoffs are computed once on the pooled noise cells of all WT donors and applied unchanged to
every donor, which is exactly what Step 1 does for WT donors. Arm B keeps `qc_pass`; Arm D adds
PBS-2 and PBS-3.

The identity-free share below is rescued cells / all Arm D cells. It equals the rescued share of
epithelium if rescued cells are epithelial as often as retained ones, and serves as a check on
Section 4.

In [ ]:
cells = assign_cell_classes(cells, reference_genotype=Genotype.WT)
added = rescued_classes(TEST_ARM, REFERENCE_ARM)
print(f"Arm {REFERENCE_ARM}: {ARMS[REFERENCE_ARM]}  |  Arm {TEST_ARM}: {ARMS[TEST_ARM]}  |  rescued: {added}")

display(cells.groupby(["donor", "cell_class"], observed=True).size().unstack(fill_value=0))

qc_only_share = rescued_share_by_donor(cells, reference_arm=REFERENCE_ARM, test_arm=TEST_ARM)
qc_only_share.style.format({c: "{:.1%}" for c in qc_only_share.columns if c.endswith("share") or c.startswith("share_")})

## 3. Provisional cell identity (stand-in for Step 2)

For every Arm D cell of each WT donor, counts are scaled to 10,000 per cell, log-transformed and
scored as mean expression of each signature's up genes minus its down genes (basal, LP, ML,
stromal). Nothing in that score depends on other cells. Scores are then z-scored with the mean and
SD of **Arm B cells only**, and each cell takes its highest-scoring type, or `other` if no z-score
is positive. Rescued cells therefore never influence what counts as LP.

Scores are cached per cell; the first run loads each WT donor's AnnData once.

In [ ]:
arm_cells = cells.loc[cells["cell_class"].isin(ARMS[TEST_ARM])].reset_index(drop=True)

if SCORES_PATH.exists() and not FORCE_RESCORE:
    scores = pd.read_csv(SCORES_PATH, dtype={"donor": str, "barcode": str})
else:
    from signals_in_the_noise.preprocessing.gse161529 import GSE161529

    gse = GSE161529()
    signatures = load_lim_signatures(excluded_genes=AMBIENT_CORE_MARKERS)
    frames = []
    for donor in WT_DONORS:
        adata = gse.get_dataset(donor.h5ad_filename)
        barcodes = arm_cells.loc[arm_cells["donor"] == donor.author_id, "barcode"].to_numpy()
        donor_scores = score_signatures(adata[barcodes], signatures)
        frames.append(donor_scores.rename_axis("barcode").reset_index().assign(donor=donor.author_id))
        del adata
    scores = pd.concat(frames, ignore_index=True)
    scores.to_csv(SCORES_PATH, index=False)

scores["donor"] = scores["donor"].astype(arm_cells["donor"].dtype)
arm_cells = arm_cells.merge(scores, on=["donor", "barcode"], how="left", validate="one_to_one")
assert arm_cells[SIGNATURE_NAMES].notna().all().all(), "Some Arm D cells have no signature scores."

in_reference_arm = arm_cells["cell_class"].isin(ARMS[REFERENCE_ARM])
arm_cells["cell_type"] = call_cell_types(arm_cells[SIGNATURE_NAMES], in_reference_arm)

Cell-type composition per donor, for retained (`qc_pass`) and rescued cells separately. Check before
reading further: every donor should have all three epithelial types among retained cells. A donor
with almost no epithelium, or with nearly everything called one type, points to a scoring problem
rather than biology.

In [ ]:
composition = pd.crosstab(
    [arm_cells["donor"], arm_cells["cell_class"]], arm_cells["cell_type"], normalize="index"
)
composition.style.format("{:.1%}").background_gradient(axis=None, cmap="Blues")

## 4. Δ per WT donor

`delta_by_donor` returns p (`lp_fraction_B`), r (`rescued_lp_fraction`), w (`rescued_share`, split
into `share_pbs-2` and `share_pbs-3`) and Δ. The assertion checks the identity Δ = w · (r − p).
The last two columns compare w with the identity-free share from Section 2; large gaps mean rescued
cells are epithelial much more or less often than retained ones.

In [ ]:
deltas = delta_by_donor(arm_cells, reference_arm=REFERENCE_ARM, test_arm=TEST_ARM)
p = deltas[f"lp_fraction_{REFERENCE_ARM}"]
r = deltas["rescued_lp_fraction"]
w = deltas["rescued_share"]
assert np.allclose(deltas["delta"], w * (r.fillna(0.0) - p)), "Δ ≠ w (r - p)"

deltas["identity_free_share"] = deltas["donor"].map(
    qc_only_share.set_index("donor")["rescued_share"]
).astype(float)
deltas.to_csv(DESIGN_DIR / "wt_delta_by_donor.csv", index=False)

percent = [f"lp_fraction_{REFERENCE_ARM}", f"lp_fraction_{TEST_ARM}", "rescued_lp_fraction",
           "rescued_share", *[f"share_{c}" for c in added], "identity_free_share"]
deltas.style.format({**{c: "{:.1%}" for c in percent}, "delta": "{:+.4f}"})

In [ ]:
x = np.arange(len(deltas))
labels = deltas["donor"].astype(str)
wt_colour = GENOTYPE_PALETTE[Genotype.WT]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bottom = np.zeros(len(deltas))
for cell_class, colour in zip(added, ["#9ecae1", "#3182bd"]):
    share = deltas[f"share_{cell_class}"].to_numpy()
    axes[0].bar(x, share, bottom=bottom, color=colour, label=cell_class.upper())
    bottom += share
axes[0].set(ylabel=f"rescued share of Arm {TEST_ARM} epithelium (w)", title="Ceiling on |Δ| per donor")
axes[0].legend()

axes[1].vlines(x, p, r, color="grey")
axes[1].scatter(x, p, color=wt_colour, label=f"retained, Arm {REFERENCE_ARM} (p)", zorder=3)
axes[1].scatter(x, r, color="black", marker="x", label="rescued (r)", zorder=3)
axes[1].set(ylabel="LP fraction of epithelium", title="LP fraction: retained vs rescued")
axes[1].legend()

mean_delta, sd_delta = deltas["delta"].mean(), deltas["delta"].std(ddof=1)
axes[2].bar(x, deltas["delta"], color=wt_colour)
axes[2].axhline(mean_delta, color="black", linestyle="--", label="WT mean")
axes[2].axhspan(mean_delta - sd_delta, mean_delta + sd_delta, color="grey", alpha=0.2, label="±1 SD")
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set(ylabel=f"Δ = LP fraction {TEST_ARM} − {REFERENCE_ARM}", title="Δ per WT donor")
axes[2].legend()

for ax in axes:
    ax.set_xticks(x, labels, rotation=90)
fig.tight_layout()
save_figure(fig, "wt_delta_by_donor")
plt.show()

## 5. SD of Δ across WT donors

The confidence interval assumes Δ is roughly normal across donors. With 8 donors it runs from about
0.66× to 2.0× the point estimate, and the MDE inherits that range.

In [ ]:
sd = sd_with_ci(deltas["delta"])
print(f"WT mean Δ = {deltas['delta'].mean():+.4f}")
print(f"SD of Δ   = {sd.sd:.4f}  (95% CI {sd.low:.4f} – {sd.high:.4f}, n = {sd.n})")

## 6. Minimum detectable effect

Power of the exact permutation test (difference in means, all 495 relabelings, reject when p ≤ α) is
simulated with the BRCA1 group shifted by a fixed effect. Donor-level Δ is either normal with the WT
SD, or resampled from the 8 WT Δ values as a check on normality. The MDE is the smallest shift in
mean Δ (BRCA1 − WT) detected with the target power.

It is judged against two benchmarks built from WT donors standing in for BRCA1 donors (similar
depth and QC per Step 0):

- **Plausible effect**: BRCA1 rescued epithelial cells are LP `PLAUSIBLE_FOLD` times as often as
  WT's, i.e. mean of w · (min(1, k·r) − r). Doubling is a generous biological effect.
- **Ceiling**: every rescued epithelial cell in a BRCA1 donor is LP, i.e. mean of w · (1 − p) minus
  the WT mean Δ. No effect carried by the rescued cells can be larger.

Arm D rescues PBS-2 and PBS-3, so both benchmarks use their combined share. If the effect were
confined to PBS-3, the relevant share is only `share_pbs-3`, which makes both benchmarks smaller
and the verdict harsher.

In [ ]:
n_donors = N_BRCA1_DONORS + len(WT_DONORS)
sd_units = np.linspace(0.0, 5.0, 201)
power_normal = simulate_power(
    sd_units, normal_errors(N_SIMULATIONS, n_donors, seed=SEED), N_BRCA1_DONORS,
    alpha=ALPHA, alternative=ALTERNATIVE,
)
mde_sd_units = minimum_detectable_effect(sd_units, power_normal, target_power=TARGET_POWER)

effects = sd_units * sd.sd
power_resampled = simulate_power(
    effects, resampled_errors(deltas["delta"], N_SIMULATIONS, n_donors, seed=SEED), N_BRCA1_DONORS,
    alpha=ALPHA, alternative=ALTERNATIVE,
)
mde_resampled = minimum_detectable_effect(effects, power_resampled, target_power=TARGET_POWER)

mde = mde_sd_units * sd.sd
ceiling = ceiling_effect(w, p, deltas["delta"])
plausible = fold_enrichment_effect(w, r, PLAUSIBLE_FOLD)

pd.DataFrame(
    {
        "value": [mde_sd_units, mde, mde_sd_units * sd.low, mde_sd_units * sd.high, mde_resampled,
                  plausible, ceiling, deltas["share_pbs-3"].mean(), w.mean()],
    },
    index=["MDE (SD units)", "MDE", "MDE at SD lower bound", "MDE at SD upper bound",
           "MDE, resampled WT Δ", f"plausible effect ({PLAUSIBLE_FOLD:g}x rescued LP rate)",
           "ceiling (all rescued epithelium LP)", "mean PBS-3 share of epithelium",
           "mean rescued share of epithelium (w)"],
).style.format("{:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(effects, power_normal, color="black", label="normal Δ")
ax.plot(effects, power_resampled, color="black", linestyle="--", label="Δ resampled from WT donors")
ax.axhline(TARGET_POWER, color="grey", linewidth=0.8)
ax.axhline(ALPHA, color="grey", linewidth=0.8, linestyle=":")
for value, label, colour in [
    (mde, f"MDE = {mde:.4f}", "black"),
    (plausible, f"plausible = {plausible:.4f}", "#2ca02c"),
    (ceiling, f"ceiling = {ceiling:.4f}", "#d62728"),
]:
    ax.axvline(value, color=colour, linewidth=1.5, alpha=0.8, label=label)
ax.set(
    xlabel="true difference in mean Δ, BRCA1 − WT",
    ylabel="power",
    title=f"Exact permutation test, {N_BRCA1_DONORS} vs {len(WT_DONORS)} donors, α = {ALPHA:g} ({ALTERNATIVE})",
    ylim=(0, 1.02),
)
ax.legend(loc="lower right")
save_figure(fig, "power_curve")
plt.show()

## 7. Which confidence interval to report

Step 5 plans a donor-bootstrap 95% CI. With 4 BRCA1 donors a percentile bootstrap is too narrow: a
resample of 4 donors has only 35 distinct outcomes, and it ignores the uncertainty in the SD. The
simulation below measures coverage for this design with no data involved. If the bootstrap falls
clearly short of 95%, a CI-based statement such as "we can exclude effects larger than X" would
overclaim with it. In that case report the pooled-variance t interval as the primary CI and the
bootstrap alongside. The exclusion bound in the decision uses the t interval.

In [ ]:
coverage = simulate_ci_coverage(
    N_BRCA1_DONORS, len(WT_DONORS), n_simulations=2000, n_resamples=2000, seed=SEED
)
coverage["expected_half_width"] = coverage["mean_half_width_sd_units"] * sd.sd
coverage.style.format(
    {"coverage": "{:.1%}", "nominal": "{:.0%}", "mean_half_width_sd_units": "{:.2f}", "expected_half_width": "{:.4f}"}
)

## 8. Decision

Recorded now, from WT donors only. Step 5 must load this file rather than recompute the framing
after BRCA1 Δ is known.

In [ ]:
decision = DesignDecision(
    n_test_donors=N_BRCA1_DONORS,
    n_reference_donors=len(WT_DONORS),
    delta_sd=sd.sd,
    delta_sd_low=sd.low,
    delta_sd_high=sd.high,
    alpha=ALPHA,
    alternative=ALTERNATIVE,
    target_power=TARGET_POWER,
    mde_sd_units=mde_sd_units,
    mde=mde,
    mde_low=mde_sd_units * sd.low,
    mde_high=mde_sd_units * sd.high,
    mde_resampled=mde_resampled,
    ceiling_effect=ceiling,
    plausible_effect=plausible,
    plausible_fold=PLAUSIBLE_FOLD,
    exclusion_bound=t_interval_half_width(sd.sd, N_BRCA1_DONORS, len(WT_DONORS)),
    identity_method="provisional: Lim et al. (2009) signatures standardised on Arm B cells; no ambient filter",
)
decision.to_json(DESIGN_DIR / "design_decision.json")
print(decision.summary())

### How Step 5 reports the result

**If the verdict is ESTIMATION**, the report opens with it, for example: *"With 4 BRCA1 and 8 WT
donors, the smallest difference in mean Δ detectable with 80% power is X (Y SD of Δ), larger than
the Z expected even if BRCA1 rescued cells were LP twice as often as WT's. We therefore estimate the
difference rather than test it."* The main result is then the difference in mean Δ with its 95% CI
(t interval, per Section 7). A null result is stated as *"the data exclude effects larger than U"*,
where U is the **observed** upper CI limit from Step 5; `exclusion_bound` here is only its expected
value under no effect. The permutation p-value is still reported, as a secondary number.

**If the verdict is HYPOTHESIS TEST**, the Step 6 decision rules apply as planned, and the CI is
still reported.

**Report back after running:** the composition table (Section 3), the Δ table and figure
(Section 4), the SD line (Section 5), the MDE table and power curve (Section 6), the coverage table
(Section 7) and the decision printout above.